## Notebook del Processing Job
El notebook debe cubrir:

- Setup: sesión de SageMaker, IAM role, bucket y prefix.
- Carga del dataset a S3: sube tus datos crudos al bucket de SageMaker.
- Ejecución del Processing Job.
- Inspección del output: lee las primeras filas del CSV transformado desde S3 para verificar que el job fue exitoso.


In [ ]:
## Setup: SageMaker Session, IAM Role y Bucket S3
from time import gmtime, strftime
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
timestamp_prefix = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

prefix = "sagemaker/processing-demo"

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

# S3 paths para el Processing Job
input_prefix = prefix + "/input"
output_prefix = prefix + "/output"

# Rutas dentro del container del Processing Job
input_container_path = "/opt/ml/processing/input"
output_container_path = "/opt/ml/processing/output"

print(f"Bucket: {bucket}")
print(f"Input S3 path: s3://{bucket}/{input_prefix}")
print(f"Output S3 path: s3://{bucket}/{output_prefix}")

### Descarga del dataset y carga a Amazon Simple Storage Service (Amazon S3)

In [ ]:
import boto3
import pandas as pd

# Ruta local de los datos
local_data_path = "../../../data/raw"
s3 = boto3.client("s3")

region = sagemaker_session.boto_region_name
f"s3://{bucket}/{input_prefix}"
#input_data = "s3://sagemaker-sample-data-{}/{input_prefix}".format(region)
input_data = f"s3://{bucket}/{input_prefix}".format(region)
print(f"Usando ruta S3: {input_data}")
!aws s3 cp $input_data .

# Uploading the training data to S3
sagemaker_session.upload_data(
    path="datos_entreno.parquet",
    bucket=bucket,
    key_prefix=input_prefix)

In [ ]:
# ## Cargar datos desde local a S3
# import os

# # Ruta local de los datos
# local_data_path = "../../../data/raw"

# # Subir datos a S3
# if os.path.exists(local_data_path):
#     print(f"Subiendo datos desde {local_data_path} a S3...")
#     s3_data_uri = sagemaker_session.upload_data(
#         path=local_data_path,
#         bucket=bucket,
#         key_prefix=input_prefix
#     )
#     print(f"Datos subidos a: {s3_data_uri}")
# else:
#     print(f"Error: La ruta local {local_data_path} no existe")
#     s3_data_uri = f"s3://{bucket}/{input_prefix}"
#     print(f"Usando ruta S3: {s3_data_uri}")

In [ ]:
## Ejecutar Processing Job con rutas de SageMaker
from sagemaker.processing import ScriptProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

# Crear el ScriptProcessor para ejecutar el script de preprocessing
script_processor = ScriptProcessor(
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="preprocessing-job",
    image_uri="YOUR_ECR_IMAGE_URI"  # Reemplazar con tu imagen Docker en ECR
)

# Configurar inputs del Processing Job
processing_inputs = [
    ProcessingInput(
        source=s3_data_uri,
        destination=input_container_path,
        s3_data_distribution_type="FullyReplicated"
    )
]

# Configurar outputs del Processing Job
processing_outputs = [
    ProcessingOutput(
        source=output_container_path,
        destination=f"s3://{bucket}/{output_prefix}",
        s3_upload_mode="EndOfJob"
    )
]

# Argumentos para el script de preprocessing
script_args = [
    "--input-path", input_container_path,
    "--output-path", output_container_path
]

# Ejecutar el Processing Job
print("Iniciando Processing Job...")
script_processor.run(
    code="../../container/preprocess.py",  # Ruta al script de preprocessing
    inputs=processing_inputs,
    outputs=processing_outputs,
    arguments=script_args
)

print("Processing Job completado!")

In [ ]:
## Verificar outputs del Processing Job
import pandas as pd

# Crear cliente S3
s3_client = sagemaker_session.boto_session.client("s3")
output_s3_path = f"s3://{bucket}/{output_prefix}"

print(f"Archivos en {output_s3_path}:")
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=output_prefix)

if "Contents" in response:
    for obj in response["Contents"]:
        print(f"  - {obj['Key']}")
        
# Descargar y verificar el archivo procesado
output_file_key = f"{output_prefix}/datos_entreno.parquet"  # Ajustar según el archivo generado
local_output_path = "processed_data.parquet"

try:
    s3_client.download_file(bucket, output_file_key, local_output_path)
    df = pd.read_parquet(local_output_path)
    print(f"\nPrimeras filas del archivo procesado:")
    print(df.head())
    print(f"\nForma del dataset: {df.shape}")
    print(f"Columnas: {df.columns.tolist()}")
except Exception as e:
    print(f"✗ Error: {e}")